# 存储格式与表格式 (Storage & Table Formats)

## 高级数据工程师面试核心考点

本节涵盖高频考点：
- Parquet vs ORC vs Avro 对比
- Parquet 内部结构：Row Group / Column Chunk / Page
- 编码方式：Dictionary Encoding / RLE / Bit Packing
- Delta Lake vs Iceberg vs Hudi 对比
- ACID on Data Lake 实现原理
- Z-Ordering / Liquid Clustering

---
> **面试提示**：文件格式和表格式是大数据工程中的基础知识，几乎所有高级数据工程师面试都会涉及。

---
## 1. Parquet vs ORC vs Avro 对比 (高频)

### 核心概念

这三种格式代表了大数据生态中最主流的存储选择，每种都有其适用场景：

| 特性 | Parquet | ORC | Avro |
|------|---------|-----|------|
| **存储方式** | 列式 (Columnar) | 列式 (Columnar) | 行式 (Row-based) |
| **Schema 演化** | 有限支持 | 有限支持 | 完整支持 |
| **压缩效率** | 极高 | 极高 | 中等 |
| **读性能** | 极佳（聚合查询）| 极佳 | 较差（需全行扫描）|
| **写性能** | 中等 | 中等 | 极佳 |
| **Schema 随数据** | 否（Footer）| 否（Footer）| 是 |
| **流式写入** | 不适合 | 不适合 | 非常适合 |
| **适用场景** | OLAP 分析 | Hive 优化 | Kafka 流、序列化 |
| **生态系统** | Spark/Flink/Hive | Hive/Spark | Kafka/Schema Registry |
| **Predicate Pushdown** | 支持（统计信息）| 支持（Bloom Filter）| 不支持 |
| **嵌套数据** | 完整支持 | 有限支持 | 完整支持 |

### 行式 vs 列式存储布局

```
原始数据：
  ID | Name  | Age | Score
   1 | Alice |  30 |  95.0
   2 | Bob   |  25 |  87.5
   3 | Carol |  35 |  92.0

行式存储 (Avro / CSV)：
  [1, Alice, 30, 95.0] [2, Bob, 25, 87.5] [3, Carol, 35, 92.0]
  优点：整行读写快，适合事务性操作
  缺点：查询单列需扫描全部数据

列式存储 (Parquet / ORC)：
  ID:    [1, 2, 3]
  Name:  [Alice, Bob, Carol]
  Age:   [30, 25, 35]
  Score: [95.0, 87.5, 92.0]
  优点：只读取需要的列，压缩率高（相同类型数据连续存储）
  缺点：行级更新成本高，写入相对复杂
```

### 面试常见问法
- *"为什么 Parquet 适合 OLAP 但 Avro 适合 Kafka？"*
  - Parquet 列式读取减少 I/O，但写入需要缓冲整个 row group；Avro 行式写入实时可追加，Schema 随数据传输，天然适合消息流。
- *"ORC 和 Parquet 哪个更快？"*
  - 取决于场景：ORC 在 Hive 生态中优化更深（内置 Bloom Filter、轻量级索引）；Parquet 在 Spark 生态兼容性更广。

In [ ]:
# Parquet vs ORC vs Avro comparison demo using PyArrow
import pyarrow as pa
import pyarrow.parquet as pq
import io
import os
import time
import tempfile

# Create a sample dataset
num_rows = 100_000
import random
random.seed(42)

names = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'] * (num_rows // 5)
ages = [random.randint(20, 60) for _ in range(num_rows)]
scores = [round(random.uniform(60, 100), 2) for _ in range(num_rows)]
departments = ['Engineering', 'Marketing', 'Sales', 'HR', 'Finance'] * (num_rows // 5)

table = pa.table({
    'id': pa.array(range(num_rows), type=pa.int64()),
    'name': pa.array(names, type=pa.string()),
    'age': pa.array(ages, type=pa.int32()),
    'score': pa.array(scores, type=pa.float64()),
    'department': pa.array(departments, type=pa.string()),
})

print(f"Dataset: {num_rows:,} rows x {len(table.schema)} columns")
print(f"Schema: {table.schema}")

In [ ]:
# Write Parquet with different compression codecs and measure size
import pyarrow.parquet as pq

with tempfile.TemporaryDirectory() as tmpdir:
    results = {}
    
    # Parquet with different compressions
    for codec in ['NONE', 'SNAPPY', 'GZIP', 'ZSTD']:
        path = os.path.join(tmpdir, f'data_{codec.lower()}.parquet')
        
        start = time.time()
        pq.write_table(table, path, compression=codec)
        write_time = time.time() - start
        
        file_size = os.path.getsize(path)
        
        start = time.time()
        _ = pq.read_table(path, columns=['score', 'department'])
        read_time = time.time() - start
        
        results[f'Parquet ({codec})'] = {
            'size_kb': file_size / 1024,
            'write_ms': write_time * 1000,
            'read_ms': read_time * 1000,
        }
    
    print(f"{'Format':<25} {'Size (KB)':>12} {'Write (ms)':>12} {'Read 2 cols (ms)':>18}")
    print('-' * 70)
    for fmt, metrics in results.items():
        print(f"{fmt:<25} {metrics['size_kb']:>12.1f} {metrics['write_ms']:>12.1f} {metrics['read_ms']:>18.1f}")

print("\n[Note] Parquet columnar layout enables reading ONLY requested columns")
print("[Note] For Avro/CSV row format, even reading 1 column requires full row scan")

---
## 2. Parquet 内部结构：Row Group / Column Chunk / Page (高频)

### 核心架构

Parquet 文件是一个精心设计的嵌套结构，每一层都有其用途：

```
Parquet 文件结构
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
┌─────────────────────────────────────────────────────┐
│  Magic Number: "PAR1" (4 bytes)                     │
├─────────────────────────────────────────────────────┤
│  Row Group 0  (默认 128MB，水平分区)                 │
│  ┌───────────────────────────────────────────────┐  │
│  │ Column Chunk: id    ← 存储该 RG 内所有 id 值  │  │
│  │   ├─ Page 0 (Data Page, 1MB)                  │  │
│  │   │    [encoded values + repetition/def levels]│  │
│  │   ├─ Page 1 (Data Page, 1MB)                  │  │
│  │   └─ Dictionary Page (可选，存字典映射表)       │  │
│  │                                               │  │
│  │ Column Chunk: name  ← 存储所有 name 值         │  │
│  │   ├─ Dictionary Page: {0:'Alice',1:'Bob',...}  │  │
│  │   └─ Data Pages: [0,1,2,0,1,...]              │  │
│  │                                               │  │
│  │ Column Chunk: age                             │  │
│  │ Column Chunk: score                           │  │
│  └───────────────────────────────────────────────┘  │
├─────────────────────────────────────────────────────┤
│  Row Group 1  (下一个 128MB 数据块)                  │
│  └─ [同上结构]                                      │
├─────────────────────────────────────────────────────┤
│  Footer (文件元数据)                                  │
│    ├─ Schema 定义                                   │
│    ├─ Row Group 元数据                              │
│    │   ├─ 每列的 min/max 统计信息 → Predicate Pushdown│
│    │   ├─ 行数、大小、偏移量                         │
│    │   └─ 编码方式、压缩算法                         │
│    └─ Key-Value metadata                           │
├─────────────────────────────────────────────────────┤
│  Footer Length (4 bytes) + Magic Number "PAR1"     │
└─────────────────────────────────────────────────────┘

关键参数：
  Row Group Size: 默认 128MB（与 HDFS/S3 block size 对齐）
  Page Size:      默认 1MB（最小 I/O 单元）
  读文件：先读 Footer → 过滤 Row Group → 读所需 Column Chunk
```

### Predicate Pushdown（谓词下推）原理

Footer 中存储了每个 Row Group 内每列的 **min/max 统计信息**。
执行 `WHERE age > 50` 时：
1. 读取 Footer（很小，通常只有几 KB）
2. 检查每个 Row Group 的 age 列：`max < 50` → 跳过整个 Row Group（128MB！）
3. 只读取可能包含符合条件行的 Row Groups

这就是为什么**按查询列排序**数据可以极大加速 Parquet 查询。

In [ ]:
# Inspect Parquet file internal structure using PyArrow
import pyarrow.parquet as pq
import pyarrow as pa
import tempfile, os

# Create a multi-row-group parquet file
import random
random.seed(42)

data = pa.table({
    'id': list(range(10000)),
    'name': ['Alice' if i % 3 == 0 else 'Bob' if i % 3 == 1 else 'Carol' for i in range(10000)],
    'age': [random.randint(20, 60) for _ in range(10000)],
    'score': [round(random.uniform(60, 100), 2) for _ in range(10000)],
})

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'inspect.parquet')
    
    # Write with small row group size to create multiple row groups
    pq.write_table(
        data,
        path,
        row_group_size=2500,   # force 4 row groups
        compression='SNAPPY',
        write_statistics=True,
    )
    
    # Read metadata
    meta = pq.read_metadata(path)
    
    print(f"=== Parquet File Metadata ===")
    print(f"Total rows:         {meta.num_rows:,}")
    print(f"Number of Row Groups: {meta.num_row_groups}")
    print(f"Number of columns:  {meta.num_columns}")
    print(f"File size:          {os.path.getsize(path):,} bytes")
    print()
    
    for rg_idx in range(meta.num_row_groups):
        rg = meta.row_group(rg_idx)
        print(f"--- Row Group {rg_idx} ---")
        print(f"  Rows:          {rg.num_rows:,}")
        print(f"  Total bytes:   {rg.total_byte_size:,}")
        
        for col_idx in range(rg.num_columns):
            col = rg.column(col_idx)
            stats = col.statistics
            print(f"  Column '{col.path_in_schema}':")
            print(f"    Compression:   {col.compression}")
            if stats:
                print(f"    Min:           {stats.min}")
                print(f"    Max:           {stats.max}")
                print(f"    Null count:    {stats.null_count}")
        print()

In [ ]:
# Demonstrate Predicate Pushdown with row group skipping
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.compute as pc
import tempfile, os, time

# Create sorted data to maximize predicate pushdown effectiveness
n = 500_000
sorted_data = pa.table({
    'age':   pa.array(sorted(range(n)) , type=pa.int32()),
    'value': pa.array(range(n), type=pa.int64()),
})

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'sorted.parquet')
    pq.write_table(sorted_data, path, row_group_size=50_000, compression='SNAPPY')
    
    meta = pq.read_metadata(path)
    print(f"Total Row Groups: {meta.num_row_groups}")
    print(f"Age range per RG (first 3):")
    for i in range(min(3, meta.num_row_groups)):
        stats = meta.row_group(i).column(0).statistics
        print(f"  RG {i}: age [{stats.min} .. {stats.max}]")
    print("  ...")
    
    # Without pushdown: read all
    start = time.perf_counter()
    full = pq.read_table(path)
    t_full = time.perf_counter() - start
    result_full = full.filter(pc.greater(pc.field('age'), 490_000))
    
    # With pushdown: skip row groups via filters parameter
    start = time.perf_counter()
    pushed = pq.read_table(path, filters=[('age', '>', 490_000)])
    t_pushed = time.perf_counter() - start
    
    print(f"\nQuery: age > 490,000")
    print(f"Without pushdown: {t_full*1000:.1f} ms, rows scanned: {len(full):,}")
    print(f"With pushdown:    {t_pushed*1000:.1f} ms, rows scanned: {len(pushed):,}")
    print(f"Speedup: {t_full/t_pushed:.1f}x (skipped most row groups!)")

---
## 3. Parquet 编码方式：Dictionary / RLE / Bit Packing (重要)

### 编码策略

Parquet 在列式存储的基础上，对每个 Column Chunk 内的数据进行编码，进一步压缩体积：

```
编码层次（从外到内）：
  压缩算法 (Snappy/Zstd/Gzip)   ← 最外层，针对字节流
    └── 编码方式                  ← 针对值的语义
          ├── Dictionary Encoding  ← 低基数列（如枚举值）
          ├── RLE (Run-Length)     ← 连续重复值
          ├── Bit Packing          ← 小整数
          └── Delta Encoding       ← 单调递增序列

Dictionary Encoding 示例：
  原始: [Alice, Bob, Alice, Carol, Alice, Bob, Bob, Alice]
  字典: {0: Alice, 1: Bob, 2: Carol}
  编码: [0, 1, 0, 2, 0, 1, 1, 0]  ← 用整数代替字符串
  节省: 字符串 → 整数，后续可继续 RLE/Bit Pack

RLE (Run-Length Encoding) 示例：
  原始: [0, 0, 0, 0, 0, 1, 1, 1, 2, 2]
  RLE:  [(0, 5), (1, 3), (2, 2)]  ← (值, 重复次数)
  节省: 10个元素 → 3对，压缩率 70%

Bit Packing 示例：
  原始: [0, 1, 2, 3]  值范围 0-3，只需 2 bits 表示
  标准: 每个 int32 = 32 bits
  打包: 4个数字 = 4 × 2 bits = 8 bits = 1 byte（原来 16 bytes）
  节省: 93.75%
```

In [ ]:
# Simulate Dictionary Encoding, RLE, and Bit Packing in Python
import struct
from itertools import groupby

# ── 1. Dictionary Encoding ──────────────────────────────────────────────────
def dict_encode(values):
    """Replace repeated string values with integer indices."""
    dictionary = {}
    encoded = []
    for v in values:
        if v not in dictionary:
            dictionary[v] = len(dictionary)
        encoded.append(dictionary[v])
    return dictionary, encoded

def dict_decode(dictionary, encoded):
    reverse = {v: k for k, v in dictionary.items()}
    return [reverse[i] for i in encoded]

raw_names = ['Alice','Bob','Alice','Carol','Alice','Bob','Bob','Alice','Carol','Alice']
dictionary, encoded_names = dict_encode(raw_names)
print("=== Dictionary Encoding ===")
print(f"Original: {raw_names}")
print(f"Dictionary: {dictionary}")
print(f"Encoded:  {encoded_names}")
# Rough size comparison
orig_bytes = sum(len(s) for s in raw_names)
dict_bytes = sum(len(k) for k in dictionary) + len(encoded_names)  # dict + 1 byte per index
print(f"Original size: ~{orig_bytes} bytes | Encoded size: ~{dict_bytes} bytes")
print(f"Decoded:  {dict_decode(dictionary, encoded_names)}")

print()

# ── 2. Run-Length Encoding (RLE) ────────────────────────────────────────────
def rle_encode(values):
    """Compress consecutive repeated values into (value, count) pairs."""
    return [(val, sum(1 for _ in group)) for val, group in groupby(values)]

def rle_decode(encoded):
    return [val for val, count in encoded for _ in range(count)]

# RLE works best on the dict-encoded result (sorted/grouped similar values)
raw_indices = [0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 0, 0, 1, 1, 1]
rle_encoded = rle_encode(raw_indices)
print("=== Run-Length Encoding (RLE) ===")
print(f"Original ({len(raw_indices)} values): {raw_indices}")
print(f"RLE encoded ({len(rle_encoded)} pairs): {rle_encoded}")
print(f"Compression ratio: {len(raw_indices)/len(rle_encoded):.1f}x")
print(f"Decoded: {rle_decode(rle_encoded)}")

print()

# ── 3. Bit Packing ──────────────────────────────────────────────────────────
def bits_needed(max_val):
    """Minimum bits needed to represent max_val."""
    return max(1, max_val.bit_length())

def bit_pack(values):
    """Pack small integers using minimum bits (simplified simulation)."""
    max_val = max(values)
    n_bits = bits_needed(max_val)
    packed_bits = ''.join(format(v, f'0{n_bits}b') for v in values)
    # Pad to full bytes
    while len(packed_bits) % 8 != 0:
        packed_bits += '0'
    packed_bytes = bytes(int(packed_bits[i:i+8], 2) for i in range(0, len(packed_bits), 8))
    return packed_bytes, n_bits

values_to_pack = [0, 1, 2, 3, 1, 0, 2, 1, 3, 0]  # range 0-3, needs 2 bits each
packed, n_bits = bit_pack(values_to_pack)
standard_size = len(values_to_pack) * 4  # int32 = 4 bytes each
print("=== Bit Packing ===")
print(f"Values: {values_to_pack}")
print(f"Max value: {max(values_to_pack)} → needs {n_bits} bits per value")
print(f"Standard int32 size: {standard_size} bytes")
print(f"Bit-packed size:     {len(packed)} bytes")
print(f"Space saving: {(1 - len(packed)/standard_size)*100:.1f}%")

---
## 4. Delta Lake vs Apache Iceberg vs Apache Hudi 对比 (高频)

### 为什么需要表格式？

原始 Parquet 文件在数据湖上的痛点：
- 无 ACID 事务：并发写入会导致数据损坏
- 无时间旅行：误删数据无法恢复
- Schema 演化困难：加列需要重写所有文件
- 分区变更代价极高

三大开源表格式（Open Table Formats）解决了上述问题：

### Delta Lake 架构

```
Delta Lake 目录结构：
  my_table/
  ├── _delta_log/                    ← 事务日志（核心！）
  │   ├── 00000000000000000000.json  ← 第0次提交 (add files)
  │   ├── 00000000000000000001.json  ← 第1次提交 (remove old, add new)
  │   ├── 00000000000000000002.json  ← 第2次提交
  │   ├── ...                        
  │   └── 00000000000000000010.checkpoint.parquet  ← 每10次提交生成检查点
  ├── part-00000-xxx.snappy.parquet  ← 实际数据文件
  ├── part-00001-xxx.snappy.parquet
  └── ...

JSON commit 内容示例：
  {"commitInfo": {"timestamp": 1700000000, "operation": "WRITE"}}
  {"add": {"path": "part-001.parquet", "size": 1024, "stats": {...}}}
  {"remove": {"path": "part-000.parquet", "deletionTimestamp": 1700000000}}

时间旅行：指定版本号 → 重放对应版本的 _delta_log → 得到该时刻的文件列表
ACID：乐观并发控制，写入时检查日志版本，冲突则重试
```

### Apache Iceberg 架构

```
Iceberg 文件层次（Snapshot → Manifest List → Manifest → Data Files）：

  Catalog (Hive Metastore / REST / Glue)
    └── Table Metadata (metadata.json)
          ├── current-snapshot-id: 1234
          └── snapshots:
                └── Snapshot 1234
                      └── manifest-list-1234.avro   ← 清单列表
                            ├── manifest-a.avro     ← 清单文件
                            │   ├── data-001.parquet
                            │   └── data-002.parquet
                            └── manifest-b.avro
                                ├── data-003.parquet
                                └── data-004.parquet

核心特性：
  分区演化：可以直接修改分区策略，无需重写数据文件
  隐藏分区：用户写 WHERE event_date = '2024-01' 而非 WHERE date_part = '2024-01'
  行级删除：使用 delete files 记录删除，无需立即重写
  多引擎支持：Spark / Flink / Trino / Hive / Presto 均可读写
```

### Apache Hudi 架构

```
Hudi 两种表类型：

  Copy-on-Write (CoW)：
    写入时直接更新 Parquet 文件（重写包含更新行的文件）
    读性能极佳 | 写放大较高 | 适合读多写少

  Merge-on-Read (MoR)：
    增量写入 delta log 文件（Avro 格式）
    定期 compaction 合并 delta → base Parquet 文件
    读时需要合并 base + delta | 写速度极快 | 适合流式写入

  Hudi Timeline（时间轴）：
    .hoodie/timeline/
      ├── 20240101120000.commit        ← 提交完成
      ├── 20240101120000.commit.requested ← 提交请求
      ├── 20240102.compaction.requested
      └── 20240102.compaction          ← Compaction 完成

  Record-level UPSERT：
    以 record key 标识每条记录，精确更新/插入
    适合 CDC (Change Data Capture) 场景
```

### 三者对比总结

| 特性 | Delta Lake | Apache Iceberg | Apache Hudi |
|------|-----------|----------------|-------------|
| **ACID 实现** | 乐观并发+事务日志 | 快照隔离 | 时间轴+MVCC |
| **时间旅行** | 支持（版本/时间戳）| 支持（快照 ID）| 支持（时间点）|
| **分区演化** | 有限（需重写）| 完整支持（核心特性）| 有限 |
| **隐藏分区** | 不支持 | 支持 | 不支持 |
| **行级 UPSERT** | 支持（MERGE）| 支持（v2）| 原生支持（核心特性）|
| **流式写入** | 支持 | 支持 | 极佳（MoR）|
| **Compaction** | OPTIMIZE | REWRITE | 自动（MoR）|
| **主要生态** | Databricks/Spark | 全引擎通用 | Spark/Flink |
| **Schema 演化** | 支持 | 支持 | 支持 |
| **适用场景** | Databricks 用户 | 多引擎/云原生 | CDC/流式更新 |

In [ ]:
# Simulate Delta Lake transaction log structure
import json
from datetime import datetime, timedelta

class DeltaLakeSimulator:
    """Simplified simulation of Delta Lake's transaction log mechanism."""
    
    def __init__(self):
        self.transaction_log = []  # Simulates _delta_log/
        self.version = -1
    
    def _commit(self, operations: list) -> int:
        """Append a new version to the transaction log."""
        self.version += 1
        entry = {
            'version': self.version,
            'timestamp': (datetime(2024, 1, 1) + timedelta(hours=self.version)).isoformat(),
            'operations': operations
        }
        self.transaction_log.append(entry)
        return self.version
    
    def write(self, files_added: list, files_removed: list = None):
        ops = [{'add': f} for f in files_added]
        if files_removed:
            ops += [{'remove': f} for f in files_removed]
        return self._commit(ops)
    
    def get_snapshot(self, version=None):
        """Replay log to get the set of active files at a given version (time travel)."""
        target = version if version is not None else self.version
        active_files = set()
        for entry in self.transaction_log:
            if entry['version'] > target:
                break
            for op in entry['operations']:
                if 'add' in op:
                    active_files.add(op['add'])
                elif 'remove' in op:
                    active_files.discard(op['remove'])
        return active_files
    
    def print_log(self):
        print("=== Delta Lake Transaction Log (_delta_log/) ===")
        for entry in self.transaction_log:
            fname = f"{entry['version']:020d}.json"
            suffix = " [checkpoint]" if entry['version'] % 10 == 0 and entry['version'] > 0 else ""
            print(f"  {fname}{suffix}  @ {entry['timestamp']}")
            for op in entry['operations']:
                action = 'ADD   ' if 'add' in op else 'REMOVE'
                fname_op = op.get('add', op.get('remove'))
                print(f"    {action}: {fname_op}")

# Simulate a sequence of operations
delta = DeltaLakeSimulator()

# Version 0: Initial write
delta.write(['part-0000.parquet', 'part-0001.parquet'])

# Version 1: Append new data
delta.write(['part-0002.parquet', 'part-0003.parquet'])

# Version 2: UPDATE → remove old file, add new file with updated rows
delta.write(['part-0000-updated.parquet'], files_removed=['part-0000.parquet'])

# Version 3: DELETE some records
delta.write(['part-0001-filtered.parquet'], files_removed=['part-0001.parquet'])

delta.print_log()

print("\n=== Time Travel (Snapshot at each version) ===")
for v in range(4):
    snapshot = delta.get_snapshot(version=v)
    print(f"  Version {v}: {sorted(snapshot)}")

---
## 5. ACID on Data Lake 实现原理 (高频)

### 传统数据库 ACID vs 数据湖 ACID

```
传统 RDBMS ACID：
  Atomicity:   Write-Ahead Log (WAL)
  Consistency: Constraint checking
  Isolation:   Lock-based concurrency control
  Durability:  WAL flush to disk

数据湖 ACID（以 Delta Lake 为例）：
  Atomicity:   原子性写入 → 文件要么全部出现在 commit 中，要么不出现
               （利用对象存储的原子性 PUT 操作）
  Consistency: Schema 验证，约束检查（Delta 3.x+）
  Isolation:   乐观并发控制（OCC）
               写入前检查 _delta_log 版本号
               如果有冲突（其他人先写）→ 读取最新版本，尝试合并，重试
  Durability:  数据文件写入对象存储（S3/GCS/ADLS）后才提交 log

乐观并发控制（OCC）流程：
  ┌─────────────────────────────────────────────┐
  │  Writer A                  Writer B          │
  │  读取 version=5             读取 version=5    │
  │  执行写入操作               执行写入操作       │
  │  写数据文件                 写数据文件         │
  │  CAS: version 5→6 ✓        CAS: version 5→6 ✗│
  │  提交成功                   检测到冲突!        │
  │                            读取 version=6    │
  │                            检查是否可以合并   │
  │                            重试提交 6→7 ✓   │
  └─────────────────────────────────────────────┘

Iceberg 的快照隔离：
  每次写入创建新的 Snapshot，旧 Snapshot 保留直到过期
  读操作基于特定 Snapshot，不受并发写入影响
  通过 Catalog 的 CAS（Compare-And-Swap）保证原子性
```

In [ ]:
# Simulate Optimistic Concurrency Control (OCC) in Delta Lake
import threading
import time
import random

class OptimisticConcurrencySimulator:
    """Simulates Delta Lake's optimistic concurrency control."""
    
    def __init__(self):
        self.current_version = 0
        self.lock = threading.Lock()  # Only used for atomic version increment (simulating S3 CAS)
        self.log = []
        self.conflicts = 0
        self.successes = 0
    
    def _atomic_commit(self, expected_version, writer_id, data):
        """Simulate atomic Compare-And-Swap on the version number."""
        with self.lock:
            if self.current_version != expected_version:
                return False  # Conflict detected!
            self.current_version += 1
            self.log.append({
                'version': self.current_version,
                'writer': writer_id,
                'data': data,
            })
            return True
    
    def write(self, writer_id, data):
        max_retries = 5
        for attempt in range(max_retries):
            # Step 1: Read current version (optimistic read)
            read_version = self.current_version
            
            # Step 2: Simulate doing work (data processing, writing data files)
            time.sleep(random.uniform(0.001, 0.01))
            
            # Step 3: Attempt to commit
            success = self._atomic_commit(read_version, writer_id, data)
            
            if success:
                self.successes += 1
                print(f"  Writer {writer_id}: SUCCESS on attempt {attempt+1}, committed version {self.current_version}")
                return
            else:
                self.conflicts += 1
                print(f"  Writer {writer_id}: CONFLICT on attempt {attempt+1}, retrying... (expected v{read_version}, actual v{self.current_version})")
                time.sleep(random.uniform(0.001, 0.005))  # Backoff
        
        print(f"  Writer {writer_id}: FAILED after {max_retries} attempts")

print("=== Optimistic Concurrency Control Simulation ===")
print("Scenario: 5 concurrent writers competing on the same Delta table\n")

sim = OptimisticConcurrencySimulator()
threads = []
for i in range(5):
    t = threading.Thread(target=sim.write, args=(f'W{i}', f'data_batch_{i}'))
    threads.append(t)

for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"\nFinal version: {sim.current_version}")
print(f"Successful commits: {sim.successes}")
print(f"Conflicts detected: {sim.conflicts}")
print("\nTransaction Log:")
for entry in sim.log:
    print(f"  v{entry['version']}: Writer={entry['writer']}, Data={entry['data']}")

---
## 6. Z-Ordering & Liquid Clustering (重要)

### 为什么需要 Z-Ordering？

```
问题：多维查询的数据局部性
  SELECT * FROM events WHERE country='US' AND event_type='click'

  按 country 排序：US 行连续存储，但 event_type 分散 → 每个 RG 都要读
  按 event_type 排序：click 行连续存储，但 country 分散 → 同样问题

  Z-Order (Z-Curve / Morton Code)：
    同时按多个维度排序，使相关数据在多维空间中局部化

  2D 示意图（country, event_type 各有 4 个值）：

    event_type →  A  B  C  D
    country ↓   ┌──┬──┬──┬──┐
              US│ 0│ 1│ 4│ 5│  Z-curve 路径：0→1→2→3→4→5→6→7→...
              UK│ 2│ 3│ 6│ 7│  保证 (US,A)和(US,B)和(UK,A)和(UK,B)
              DE│ 8│ 9│12│13│  在文件中相邻存储
              FR│10│11│14│15│
                └──┴──┴──┴──┘
    查询 country='US' AND event_type IN ('A','B'):
      只需读取包含 0,1,2,3 的文件，大量跳过其他文件！

Delta Lake OPTIMIZE + ZORDER BY:
  OPTIMIZE my_table ZORDER BY (country, event_type)
  → 重排文件，使 Z-Order 相邻的行存储在同一 Row Group

Liquid Clustering（Databricks 新特性）：
  Z-Ordering 的进化版，核心改进：
  1. 增量式聚类（Incremental）：新数据不需要重写全表
  2. 自动选择聚类列（Auto Clustering）
  3. 无需手动 OPTIMIZE：后台自动触发
  4. 比 Z-Ordering 更灵活，支持更多列

  CREATE TABLE t CLUSTER BY (country, event_type);  -- Liquid Clustering DDL
```

In [ ]:
# Simulate Z-Order (Morton Code) calculation and demonstrate data co-location

def interleave_bits(x: int, y: int) -> int:
    """Compute the Z-order (Morton code) by interleaving bits of x and y."""
    result = 0
    for i in range(16):
        result |= ((x >> i) & 1) << (2 * i)
        result |= ((y >> i) & 1) << (2 * i + 1)
    return result

def z_order_encode(row: dict, columns: list, max_val: int = 255) -> int:
    """Multi-dimensional Z-order encoding for a row."""
    result = 0
    shift = 0
    for i in range(len(columns) - 1):
        a = int(row[columns[i]] * max_val / 100)
        b = int(row[columns[i+1]] * max_val / 100)
        result = interleave_bits(a, b)
        break  # simplified 2D case
    return result

# Simulate dataset with two dimensions: age (0-100) and score (0-100)
import random
random.seed(42)

data = [
    {'id': i, 'age': random.randint(0, 100), 'score': random.randint(0, 100)}
    for i in range(20)
]

# Assign Z-order values
for row in data:
    row['z_order'] = interleave_bits(row['age'], row['score'])

# Sort by Z-order
data_sorted = sorted(data, key=lambda r: r['z_order'])

print("=== Z-Ordering Simulation ===")
print(f"{'ID':>4} {'Age':>5} {'Score':>7} {'Z-Order':>12}")
print("-" * 32)
for row in data_sorted:
    print(f"{row['id']:>4} {row['age']:>5} {row['score']:>7} {row['z_order']:>12}")

# Demonstrate co-location: query age IN [20-40] AND score IN [30-60]
query_age_min, query_age_max = 20, 40
query_score_min, query_score_max = 30, 60

# Simulate row groups (5 rows each)
row_groups = [data_sorted[i:i+5] for i in range(0, len(data_sorted), 5)]

print(f"\n=== Query: age=[{query_age_min},{query_age_max}] AND score=[{query_score_min},{query_score_max}] ===")
print("Row Groups accessed:")
for rg_idx, rg in enumerate(row_groups):
    matching = [r for r in rg if query_age_min <= r['age'] <= query_age_max 
                and query_score_min <= r['score'] <= query_score_max]
    age_range = (min(r['age'] for r in rg), max(r['age'] for r in rg))
    score_range = (min(r['score'] for r in rg), max(r['score'] for r in rg))
    
    # Can we skip this row group? (using min/max stats)
    can_skip = (age_range[1] < query_age_min or age_range[0] > query_age_max or
                score_range[1] < query_score_min or score_range[0] > query_score_max)
    
    status = "SKIP" if can_skip else f"READ ({len(matching)} matches)"
    print(f"  RG {rg_idx}: age={age_range}, score={score_range} → {status}")

In [ ]:
# Compare: No ordering vs Z-Ordering for multi-dimensional queries
import random
random.seed(123)

N = 1000
dataset = [
    {'id': i, 'country_code': random.randint(0, 9), 'event_type': random.randint(0, 9)}
    for i in range(N)
]

# Assign Z-order
for row in dataset:
    row['z_order'] = interleave_bits(row['country_code'], row['event_type'])

# Two layouts: natural order vs Z-order
natural_order = dataset.copy()
z_order_layout = sorted(dataset, key=lambda r: r['z_order'])

RG_SIZE = 100  # 100 rows per row group

def count_rgs_accessed(layout, country_code, event_type):
    """Count how many row groups must be read for a 2D point query."""
    rgs = [layout[i:i+RG_SIZE] for i in range(0, len(layout), RG_SIZE)]
    accessed = 0
    for rg in rgs:
        cc_vals = [r['country_code'] for r in rg]
        et_vals = [r['event_type'] for r in rg]
        # Can we skip? Use min/max stats
        if (min(cc_vals) <= country_code <= max(cc_vals) and
            min(et_vals) <= event_type <= max(et_vals)):
            accessed += 1
    return accessed

print("=== Row Groups Accessed: Natural Order vs Z-Order ===")
print(f"{'Query (country, event_type)':<30} {'Natural':>10} {'Z-Order':>10} {'Savings':>10}")
print("-" * 62)

total_rgs = N // RG_SIZE
test_queries = [(2, 3), (5, 7), (0, 0), (9, 9), (4, 5)]
for cc, et in test_queries:
    nat = count_rgs_accessed(natural_order, cc, et)
    zord = count_rgs_accessed(z_order_layout, cc, et)
    savings = f"{(1 - zord/nat)*100:.0f}%" if nat > 0 else "N/A"
    print(f"  country={cc}, event_type={et}          {nat:>10} {zord:>10} {savings:>10}")

print(f"\n  Total row groups: {total_rgs}")
print("  Z-Order significantly reduces row groups accessed for multi-column queries!")

---
## 复习要点 (Review Summary)

### 高频考点速记

**1. 格式选择原则**
- **Parquet** = OLAP 分析 / Spark 首选 / 列式 / 高压缩
- **ORC** = Hive 生态 / 内置 Bloom Filter / Stripe 替代 Row Group
- **Avro** = Kafka 序列化 / Schema 随数据 / 行式 / 流式写入

**2. Parquet 三层结构**
- File → **Row Group** (128MB) → **Column Chunk** (一列的数据) → **Page** (1MB)
- Footer 存 min/max → **Predicate Pushdown** → 跳过整个 Row Group
- Dictionary Page → Data Pages → 低基数列压缩率极高

**3. 编码方式**
- **Dictionary**: 重复字符串 → 整数索引 (`Alice→0, Bob→1`)
- **RLE**: 连续相同值 → `(值, 次数)` 对
- **Bit Packing**: 小整数压缩至最小位宽

**4. 三大表格式核心差异**
- **Delta Lake**: `_delta_log/` JSON + Parquet checkpoint，乐观并发，Databricks 生态
- **Iceberg**: Snapshot → Manifest List → Manifest → Data Files，**分区演化**是核心优势
- **Hudi**: CoW vs MoR，**Record-level UPSERT**，CDC 场景首选

**5. ACID 实现**
- 数据湖 ACID ≠ 数据库 ACID（无锁管理器）
- 乐观并发控制：读版本 → 写数据 → CAS 提交 → 冲突重试
- Durability：先写数据文件，再写事务日志

**6. Z-Ordering**
- Morton Code 交错位：多维 → 一维，相邻多维点在文件中也相邻
- 适合多列过滤查询，但需要全表重写（OPTIMIZE 操作）
- **Liquid Clustering** = 增量式 Z-Ordering，无需全表重写

---
## 练习 (Exercises)

### 练习 1：文件格式选择 (概念)

下列场景各应选择什么文件格式？解释原因：
1. Kafka topic 存储用户行为事件，消费者用 Spark 做实时 ETL
2. 数仓 fact table，每天 10 亿行，主要做聚合查询（GROUP BY, SUM）
3. 需要频繁 Schema 变更的数据集，且下游有多种语言的消费者
4. Hive on HDFS，ORC vs Parquet 如何选择？

In [ ]:
# 练习 1 参考答案
answers = {
    1: "Avro → Schema 随消息传递，支持 Schema Registry，Kafka 生产者/消费者原生支持，行式写入性能好",
    2: "Parquet → 列式存储只读聚合列，高压缩率，Predicate Pushdown 跳过无关 Row Groups",
    3: "Avro → 完整的 Schema 演化支持，JSON-like 自描述，多语言 SDK（Java/Python/Go/C++）",
    4: "ORC → Hive 对 ORC 优化更深（ORC Vectorized Reader），内置 Bloom Filter，轻量级索引"
}

for q, a in answers.items():
    print(f"Q{q}: {a}")
    print()

### 练习 2：Parquet Row Group 计算 (计算题)

一张表有以下特征：
- 1 亿行数据
- 平均每行 200 bytes（压缩前）
- Snappy 压缩比约 3:1
- Row Group size = 128 MB

问题：
1. 压缩后文件总大小是多少？
2. 大约有多少个 Row Groups？
3. 执行 `WHERE date = '2024-01-01'`（该日期约占 1/365），最少需要读多少个 Row Groups？（假设数据按日期排序）

In [ ]:
# 练习 2 参考计算
total_rows = 100_000_000
bytes_per_row_raw = 200
compression_ratio = 3
row_group_size_mb = 128

# Q1: Total file size
total_raw_bytes = total_rows * bytes_per_row_raw
total_compressed_bytes = total_raw_bytes / compression_ratio
total_compressed_gb = total_compressed_bytes / (1024**3)
print(f"Q1: File size = {total_rows:,} * {bytes_per_row_raw} / {compression_ratio} = {total_compressed_gb:.2f} GB")

# Q2: Number of Row Groups
row_group_size_bytes = row_group_size_mb * 1024 * 1024
num_row_groups = total_compressed_bytes / row_group_size_bytes
print(f"Q2: Row Groups = {total_compressed_gb:.2f} GB / {row_group_size_mb} MB = ~{num_row_groups:.0f} Row Groups")

# Q3: Row groups needed for date filter (data sorted by date)
fraction = 1 / 365
rgs_for_one_day = num_row_groups * fraction
print(f"Q3: Date sorted → 1/365 of RGs = {num_row_groups:.0f} * {fraction:.4f} = ~{rgs_for_one_day:.0f} Row Groups")
print(f"    (vs {num_row_groups:.0f} without sorting = {365:.0f}x improvement!)")

### 练习 3：Delta Lake vs Iceberg 选型 (场景题)

你的公司正在为以下场景选择表格式，请给出建议并说明理由：

**场景 A**：公司 100% 使用 Databricks，需要 UPSERT + 时间旅行 + SQL 兼容。  
**场景 B**：多团队，分别使用 Spark（ETL）、Trino（查询）、Flink（流处理），数据湖在 S3。  
**场景 C**：需要从 MySQL CDC 实时同步到数据湖，写入频率极高（10万 UPSERT/秒）。  
**场景 D**：分区策略从按天分区改为按小时分区，但不想重写历史数据。

In [ ]:
# 练习 3 参考答案
recommendations = {
    'A': {
        'choice': 'Delta Lake',
        'reason': 'Databricks 原生支持，MERGE INTO 语法成熟，Unity Catalog 集成，OPTIMIZE/ZORDER 工具完善'
    },
    'B': {
        'choice': 'Apache Iceberg',
        'reason': '多引擎支持最佳（Spark+Trino+Flink 均有成熟 connector），REST Catalog 标准化，避免引擎锁定'
    },
    'C': {
        'choice': 'Apache Hudi (MoR)',
        'reason': 'Record-level UPSERT 原生支持，MoR 写入延迟低，Hudi Streamer 直接对接 Debezium CDC 事件'
    },
    'D': {
        'choice': 'Apache Iceberg',
        'reason': 'Partition Evolution 核心特性：直接 ALTER TABLE ... PARTITION BY (hours(ts)) 无需重写历史文件'
    },
}

for scenario, rec in recommendations.items():
    print(f"场景 {scenario}: 选择 {rec['choice']}")
    print(f"  原因: {rec['reason']}")
    print()

### 练习 4：编码方式选择 (概念)

对以下列数据，Parquet 会优先选择哪种编码方式？为什么？
1. `status` 列：值为 `['active', 'inactive', 'pending']`，有 1 亿行
2. `timestamp` 列：Unix 时间戳，单调递增，间隔约 1 秒
3. `is_deleted` 列：布尔值，99% 为 False
4. `user_id` 列：随机 UUID 字符串，无重复

In [ ]:
# 练习 4 参考答案
encoding_answers = [
    {
        'column': 'status (3 distinct values)',
        'encoding': 'Dictionary + RLE/Bit Packing',
        'reason': '低基数（cardinality=3）→ Dictionary 效果极佳，字符串→2位整数，后续 RLE 进一步压缩'
    },
    {
        'column': 'timestamp (monotonic)',
        'encoding': 'Delta Encoding (差值编码)',
        'reason': '单调递增序列，存储相邻值的差（约1000ms），差值远小于原值，大幅压缩'
    },
    {
        'column': 'is_deleted (99% False)',
        'encoding': 'RLE + Bit Packing',
        'reason': '大量连续 False → RLE (False, 10000) 效果极佳；1 bit 即可表示布尔值'
    },
    {
        'column': 'user_id (random UUID)',
        'encoding': 'Plain Encoding (无编码)',
        'reason': '高基数随机字符串，Dictionary 无效（字典会膨胀到与数据同大小），直接存储原值'
    },
]

for i, ans in enumerate(encoding_answers, 1):
    print(f"Q{i}: {ans['column']}")
    print(f"     编码: {ans['encoding']}")
    print(f"     原因: {ans['reason']}")
    print()

### 练习 5：Z-Ordering 适用场景分析 (场景题)

下表有列：`user_id, country, product_category, event_date, revenue`

常见查询模式：
- Q1: `WHERE country = 'US' AND event_date = '2024-01-01'` (80% 查询)
- Q2: `WHERE user_id = ?` (10% 查询)
- Q3: `GROUP BY country, product_category` (10% 查询)

请回答：
1. 应该 ZORDER BY 哪些列？
2. 如果改用 Liquid Clustering，有何优势？
3. Z-Ordering 的局限性是什么？

In [ ]:
# 练习 5 参考答案
print("=== 练习 5 参考答案 ===")
print()
print("Q1: 应选择 ZORDER BY (country, event_date)")
print("    - 80% 查询同时过滤 country 和 event_date")
print("    - Z-Order 确保同一 country+date 组合的数据存储在相邻 Row Groups")
print("    - user_id 查询只占 10%，且 UUID 高基数不适合 Z-Order")
print()

print("Q2: Liquid Clustering 优势")
print("    - 增量聚类：新写入数据无需全表重写，OPTIMIZE 命令只处理未聚类文件")
print("    - 自适应：可以在不重建表的情况下改变聚类列")
print("    - 后台自动：Databricks 可自动调度，无需手动触发")
print("    - 更低成本：避免了 Z-Order 全表 shuffle 的高计算成本")
print()

print("Q3: Z-Ordering 局限性")
print("    - 写放大：每次 OPTIMIZE 需要重写大量文件（全表 shuffle）")
print("    - 列数限制：超过 4 列后效果下降（Z-curve 在高维空间局部性变差）")
print("    - 点查询差：对单条记录查询（如 WHERE user_id = ?）无明显优化")
print("    - 不支持增量：新写入数据不自动聚类，需周期性 OPTIMIZE")
print("    - 与分区叠加：应先分区（粗粒度）再 Z-Order（细粒度），过度分区会抵消效果")